# Day 2 — QAT Pareto sweep

Run one copy per available GPU. Every candidate starts from the clean 45,000-example-training baseline, selects its checkpoint using the same deterministic 5,000-example validation split, and evaluates only the selected model on CIFAR-10's held-out test set.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
username, repo_name = 'AdiGiriIIT', 'CS6886--Assignment-2'
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2
%cd assignment-2

In [ ]:
from pathlib import Path

# Must be baseline.pt produced by MobileNet_Baseline.ipynb, not the earlier all-50k baseline.
BASELINE_SOURCE = '/kaggle/input/datasets/adityagirishep23b048/baseline/baseline.pt'  # update to the new baseline dataset version
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'

!python -m pip install -q PyYAML matplotlib
!mkdir -p results/checkpoints results/logs experiments/sweeps
!cp $BASELINE_SOURCE results/checkpoints/baseline.pt
!sha256sum results/checkpoints/baseline.pt
cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
required = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta']
assert all((cifar_dir / name).is_file() for name in required), f'Missing CIFAR-10 files under {cifar_dir}'
print('Using CIFAR-10 at', cifar_dir)
print('QAT split: 45,000 train / 5,000 validation (seed 6886). Test remains held out until the final cell.')

In [ ]:
# Correctness gates before consuming a full GPU run. No test evaluation occurs here.
!set -o pipefail; python -m unittest discover -s tests -v 2>&1 | tee results/logs/day2-correctness.log
!set -o pipefail; nvidia-smi 2>&1 | tee results/logs/day2-gpu.log

In [ ]:
# Change these values in each parallel notebook.
RUN_NAME = 'w8a8-seed6886'
WEIGHT_BITS = 8
ACTIVATION_BITS = 8
EPOCHS = 12

assert (WEIGHT_BITS, ACTIVATION_BITS) in {(8, 8), (6, 6), (4, 6), (4, 4)}
print(f'Launching {RUN_NAME}: W{WEIGHT_BITS}A{ACTIVATION_BITS}')

In [ ]:
# src.qat selects qat-{RUN_NAME}-best-target.pt solely from epochs at the requested precision.
!set -o pipefail; python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir "{DATA_DIR}" --device cuda --weight-bits {WEIGHT_BITS} --activation-bits {ACTIVATION_BITS} --epochs {EPOCHS} --run-name {RUN_NAME} 2>&1 | tee results/logs/{RUN_NAME}.log

In [ ]:
import csv
from IPython.display import Image, display

run_dir = Path('experiments/sweeps') / RUN_NAME
plot = run_dir / 'history.png'
assert plot.is_file(), f'Missing QAT plot: {plot}'
display(Image(filename=str(plot)))
diagnostics = run_dir / 'activation_quantization_diagnostics.csv'
assert diagnostics.is_file(), f'Missing activation diagnostics: {diagnostics}'
with diagnostics.open(newline='') as handle:
    rows = list(csv.DictReader(handle))
target_rows = [row for row in rows if row['phase'] == 'validation' and int(row['bits']) == ACTIVATION_BITS]
worst = sorted(target_rows, key=lambda row: float(row['saturation_percent']), reverse=True)[:10]
print('Most saturated target-precision validation boundaries:')
for row in worst:
    print(f"{row['quantizer']}: {row['saturation_percent']}% saturated; scale={row['scale']}; clip=[{row['clip_min']}, {row['clip_max']}]")

In [ ]:
# Final held-out test evaluation; neither result is used for selection.
!set -o pipefail; python -m src.evaluate --checkpoint results/checkpoints/baseline.pt --data-dir "{DATA_DIR}" --device cuda 2>&1 | tee results/logs/baseline-held-out-test.log
!set -o pipefail; python -m src.evaluate --checkpoint results/checkpoints/qat-{RUN_NAME}-best-target.pt --data-dir "{DATA_DIR}" --device cuda 2>&1 | tee results/logs/{RUN_NAME}-held-out-test.log

In [ ]:
import hashlib
import tarfile
from IPython.display import FileLink

run_dir = Path('experiments/sweeps') / RUN_NAME
paths = [run_dir, Path('results/logs') / f'{RUN_NAME}.log', Path('results/logs') / f'{RUN_NAME}-held-out-test.log', Path('results/checkpoints') / f'qat-{RUN_NAME}-best-target.pt', Path('results/checkpoints') / f'qat-{RUN_NAME}-latest.pt']
paths = [path for path in paths if path.exists()]
assert (run_dir / 'metrics.json').is_file(), f'Missing {run_dir}/metrics.json'
assert len(paths) >= 4, 'Expected run record, logs, and QAT checkpoints.'
for path in paths:
    if path.is_file():
        print(f'{hashlib.sha256(path.read_bytes()).hexdigest()}  {path}')
archive = Path(f'{RUN_NAME}-artifacts.tgz')
with tarfile.open(archive, 'w:gz') as tar:
    for path in paths:
        tar.add(path, arcname=str(path))
print(f'Created {archive} ({archive.stat().st_size:,} bytes)')
FileLink(str(archive))